In [ ]:
from metavision_core.event_io import EventsIterator
import h5py
import numpy as np
import pandas as pd
import os
from pathlib import Path

In [2]:
INPUT_RAW = "//172.16.222.2/EBVrecordings/2024-TrafficMonitoring/data_collection/week_34/box_3/2024_08_28_16_37_57_recordings/events.raw"
OUTPUT_DIR = Path("//172.16.222.2/tmp/Nora/traffic_flow_data/week_34/box_3/2024_08_28_16_37_57_recordings/event_data")
OUTPUT_PREFIX = os.path.join(OUTPUT_DIR, "event_clip")
timestamp_path = "//172.16.222.2/EBVrecordings/2024-TrafficMonitoring/data_collection/week_34/box_3/2024_08_28_16_37_57_recordings/timestamps.csv"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

### Convert event data into 1 minute batches of events

In [ ]:
# Load frame timestamps
df_timestamps = pd.read_csv(timestamp_path)

SHIFT = df_timestamps.index[df_timestamps["event_time"] != 0][0]

time_windows1 = df_timestamps.iloc[SHIFT:, 0].to_numpy().astype(float)
time_windows = time_windows1 - time_windows1[0]


# Parameters in frames
CLIP_LEN_FRAMES = 5400      # 1 minute at nominal 90 fps
CLIP_STEP_FRAMES = CLIP_LEN_FRAMES

# Build clip boundaries from frame timestamps
clip_ranges = []
clip_idx = 0

while True:
    start_frame = clip_idx * CLIP_STEP_FRAMES
    end_frame = start_frame + CLIP_LEN_FRAMES

    # Check end_frame exists
    if end_frame >= len(time_windows):
        break

    start_ts = time_windows[start_frame]
    end_ts = time_windows[end_frame]

    clip_label = clip_idx  # 000, 001, 002, ...

    clip_ranges.append({
        "label": clip_label,
        "start_frame": start_frame,
        "end_frame": end_frame,
        "start_ts": start_ts,
        "end_ts": end_ts,
    })

    clip_idx += 1

print(f"Prepared {len(clip_ranges)} clip windows.")


# Event dtype
event_dtype = np.dtype([
    ("x", np.int16),
    ("y", np.int16),
    ("p", np.int8),
    ("t", np.int64),
])


# Iterate through events and write separate clip files
events_iterator = EventsIterator(INPUT_RAW)

current_clip_idx = 0
events_buffer = []
c = 0

if len(clip_ranges) == 0:
    raise ValueError("No valid clip ranges could be created from the timestamp file.")

current_range = clip_ranges[current_clip_idx]

for evs in events_iterator:
    for ev in evs:
        t = int(ev["t"])

        # Skip events before the current clip starts
        if t < current_range["start_ts"]:
            continue

        # Passed the end of the current clip, write it and move on
        while t >= current_range["end_ts"]:
            out_path = f"{OUTPUT_PREFIX}_{current_range['label']:03d}.h5"

            with h5py.File(out_path, "w") as f:
                f.create_dataset("events", data=np.array(events_buffer, dtype=event_dtype))
                f.create_dataset("start_frame", data=current_range["start_frame"])
                f.create_dataset("end_frame", data=current_range["end_frame"])
                f.create_dataset("start_ts", data=current_range["start_ts"])
                f.create_dataset("end_ts", data=current_range["end_ts"])

            print("Wrote:", out_path)

            events_buffer = []
            current_clip_idx += 1

            if current_clip_idx >= len(clip_ranges):
                print(1)
                break

            current_range = clip_ranges[current_clip_idx]

            # If the current event is before the next clip start, stop advancing
            if t < current_range["end_ts"]:
                print(2)
                break

        if current_clip_idx >= len(clip_ranges):
            print(3)
            break

        # Keep event only if it falls inside current clip window
        if current_range["start_ts"] <= t < current_range["end_ts"]:
            events_buffer.append((
                int(ev["x"]),
                int(ev["y"]),
                int(ev["p"]),
                t
            ))

    if current_clip_idx >= len(clip_ranges):
        print(4)
        break

# Flush final clip if it has collected events
if current_clip_idx < len(clip_ranges) and len(events_buffer) > 0:
    current_range = clip_ranges[current_clip_idx]
    out_path = f"{OUTPUT_PREFIX}_{current_range['label']:03d}.h5"

    with h5py.File(out_path, "w") as f:
        f.create_dataset("events", data=np.array(events_buffer, dtype=event_dtype))
        f.create_dataset("start_frame", data=current_range["start_frame"])
        f.create_dataset("end_frame", data=current_range["end_frame"])
        f.create_dataset("start_ts", data=current_range["start_ts"])
        f.create_dataset("end_ts", data=current_range["end_ts"])

    print("Wrote:", out_path)

Prepared 175 clip windows.
Wrote: \\172.16.222.2\tmp\Nora\traffic_flow_data\week_34\box_3\2024_08_28_16_37_57_recordings\event_data\event_clip_000.h5
2
Wrote: \\172.16.222.2\tmp\Nora\traffic_flow_data\week_34\box_3\2024_08_28_16_37_57_recordings\event_data\event_clip_001.h5
2
Wrote: \\172.16.222.2\tmp\Nora\traffic_flow_data\week_34\box_3\2024_08_28_16_37_57_recordings\event_data\event_clip_002.h5
2
Wrote: \\172.16.222.2\tmp\Nora\traffic_flow_data\week_34\box_3\2024_08_28_16_37_57_recordings\event_data\event_clip_003.h5
2
Wrote: \\172.16.222.2\tmp\Nora\traffic_flow_data\week_34\box_3\2024_08_28_16_37_57_recordings\event_data\event_clip_004.h5
2
Wrote: \\172.16.222.2\tmp\Nora\traffic_flow_data\week_34\box_3\2024_08_28_16_37_57_recordings\event_data\event_clip_005.h5
2
Wrote: \\172.16.222.2\tmp\Nora\traffic_flow_data\week_34\box_3\2024_08_28_16_37_57_recordings\event_data\event_clip_006.h5
2
Wrote: \\172.16.222.2\tmp\Nora\traffic_flow_data\week_34\box_3\2024_08_28_16_37_57_recordings\even